In [ ]:
# CELL 1: IMPORTS AND SETUP
from __future__ import annotations
from pathlib import Path
from typing import List, Tuple, Dict, Any, Set, Optional
import math, colorsys, itertools, random, re, sys, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap, to_rgb, to_hex
from matplotlib.patches import Patch, Rectangle
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Video, display

BASE_COLOURS = {
    "NewMC": "#ffcc99", "DR": "#ffcccc", "NewMC_Expand": "#ccffcc",
    "NewMC_Restricted": "#ccccff", "NewMC_Biased": "#ffffcc",
    "NewMC_Restricted_Expand": "#ffccff", "NewMC_Biased_Expand": "#ccffff",
    "Kawasaki": "#ffcccc", "Kawasaki_Black": "#000000",
}

def get_distinct_colors(n):
    colors = ["#000000"]
    if n > 1:
        for i in range(n - 1):
            hue = (i / (n - 1)) * 0.8 + 0.1
            saturation = 0.8
            value = 0.75
            rgb = colorsys.hsv_to_rgb(hue, saturation, value)
            colors.append(to_hex(rgb))
    return colors

def parse_grid_config_string_to_species_array(config_raw: any, N: int) -> Optional[np.ndarray]:
    if pd.isna(config_raw) or str(config_raw).strip() in ["", '""', "''"]: return None
    s = str(config_raw).strip('"\' ')
    if not s: return None
    parts = s.split(';')
    if len(parts) != N * N: return None
    species = np.zeros((N, N), dtype=np.int8)
    try:
        for i, cell_data_str in enumerate(parts):
            if not cell_data_str.strip(): continue
            cell_parts = cell_data_str.split('|')
            if len(cell_parts) == 0: continue
            species_val = int(cell_parts[0])
            y, x = i // N, i % N
            species[y, x] = species_val
    except (ValueError, IndexError): return None
    return species

In [ ]:
# CELL 2: DRIVER FUNCTIONS FOR VISUALIZATION TYPES
def energy_analysis_driver(csv_files, table_labels, curve_colours, title="Energy Analysis"):
    plt.rcParams.update({'font.size': 13})
    fig = plt.figure(figsize=(20, 10))
    
    ax1 = fig.add_subplot(2, 2, 1)
    ax2 = fig.add_subplot(2, 2, 2)
    ax3 = fig.add_subplot(2, 1, 2)
    
    fig.suptitle(title, fontsize=16, fontweight="bold")
    
    for fp, lbl, col in zip(csv_files, table_labels, curve_colours):
        if not Path(fp).is_file(): continue
        try:
            df = pd.read_csv(fp)
            if "Energy" not in df.columns: continue
            
            steps = np.arange(len(df))
            E = df["Energy"].ffill().bfill()
            
            ax1.plot(steps, E, lw=1, color=col, alpha=0.7, label=lbl)
            
            cum = E.expanding().mean()
            ax2.plot(steps, cum, lw=2, color=col, alpha=0.8)
            
            if "Total_Acceptances" in df.columns:
                ax3.plot(steps, df["Total_Acceptances"], color=col, label=lbl)
                
        except Exception: continue
    
    ax1.set(xlabel="Iteration", ylabel="Energy")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.set(xlabel="Iteration", ylabel="Cumulative Energy")
    ax2.grid(True, alpha=0.3)
    
    ax3.set(xlabel="Iteration", ylabel="Total Acceptances")
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def cluster_analysis_driver(csv_files, table_labels, curve_colours):
    for csv_file, table_label in zip(csv_files, table_labels):
        if not Path(csv_file).is_file(): continue
        
        try:
            df = pd.read_csv(csv_file)
            boundary_cols = [col for col in df.columns if col.startswith('BoundarySize_Cluster')]
            if not boundary_cols: continue
            
            fig, ax = plt.subplots(figsize=(12, 8))
            iterations = np.arange(len(df))
            
            for col in boundary_cols:
                cluster_num = col.replace('BoundarySize_Cluster', '')
                boundary_data = df[col].fillna(0)
                interior_col = f'InteriorSize_Cluster{cluster_num}'
                
                if interior_col in df.columns:
                    interior_data = df[interior_col].fillna(0)
                    total_size = boundary_data + interior_data
                    
                    ax.plot(iterations, boundary_data, label=f'Cluster {cluster_num} Boundary', linestyle='-')
                    ax.plot(iterations, interior_data, label=f'Cluster {cluster_num} Interior', linestyle='--')
                    ax.plot(iterations, total_size, label=f'Cluster {cluster_num} Total', linestyle=':')
            
            ax.set_title(f'Cluster Analysis: {table_label}')
            ax.set_xlabel('Iteration')
            ax.set_ylabel('Cluster Size')
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        except Exception: continue

def grid_visualization_driver(csv_files, table_labels):
    for csv_file, table_label in zip(csv_files, table_labels):
        config_file = csv_file.replace('.csv', '_GridConfig.csv')
        if not Path(config_file).is_file(): continue
        
        try:
            cfg_df = pd.read_csv(config_file)
            if cfg_df.empty: continue
            
            required_cols = ["x", "y", "species", "cluster_id", "cell_type"]
            if not all(col in cfg_df.columns for col in required_cols): continue
            
            N = int(max(cfg_df["x"].max(), cfg_df["y"].max()) + 1)
            
            fig, ax = plt.subplots(figsize=(10, 10))
            ax.set(xlim=(-0.5, N - 0.5), ylim=(-0.5, N - 0.5), aspect="equal")
            ax.set_title(f"Grid Configuration: {table_label}")
            
            for _, r in cfg_df.iterrows():
                color = "#d62728" if r["species"] == 0 else "#1f77b4"
                alpha = 0.4 if r["cell_type"] == "INTERIOR" else 0.8
                
                rect = Rectangle((r["x"] - 0.4, r["y"] - 0.4), 0.8, 0.8,
                               facecolor=color, alpha=alpha, edgecolor='black')
                ax.add_patch(rect)
                ax.text(r["x"], r["y"], str(r["cluster_id"]), ha="center", va="center")
            
            plt.tight_layout()
            plt.show()
        except Exception: continue

def moves_analysis_driver(csv_files, table_labels, curve_colours):
    for csv_file, table_label, color in zip(csv_files, table_labels, curve_colours):
        if not Path(csv_file).is_file(): continue
        
        try:
            df = pd.read_csv(csv_file)
            required_cols = ["Total_Acceptances", "DeltaE", "NumAtomsSwapped"]
            if not all(col in df.columns for col in required_cols): continue
            
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            fig.suptitle(f'Move Analysis: {table_label}')
            
            iterations = np.arange(len(df))
            
            axes[0, 0].plot(iterations, df["Total_Acceptances"], color=color)
            axes[0, 0].set_title('Total Acceptances')
            axes[0, 0].grid(True, alpha=0.3)
            
            axes[0, 1].plot(iterations, df["NumAtomsSwapped"], color=color)
            axes[0, 1].set_title('Atoms Swapped')
            axes[0, 1].grid(True, alpha=0.3)
            
            axes[1, 0].plot(iterations, df["DeltaE"], color=color)
            axes[1, 0].set_title('Energy Change')
            axes[1, 0].grid(True, alpha=0.3)
            
            if "ForwardProb" in df.columns and "ReverseProb" in df.columns:
                ratio = df["ForwardProb"] / df["ReverseProb"].replace(0, np.nan)
                axes[1, 1].plot(iterations, ratio, color=color)
                axes[1, 1].set_title('Forward/Reverse Prob Ratio')
                axes[1, 1].set_yscale('log')
                axes[1, 1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        except Exception: continue

def animation_driver(csv_files, table_labels, start_iteration=0, end_iteration=50, interval=200):
    for csv_file, table_label in zip(csv_files, table_labels):
        if not Path(csv_file).is_file(): continue
        
        try:
            df = pd.read_csv(csv_file)
            if "GridConfigStarting" not in df.columns or df.empty: continue
            
            df_subset = df.iloc[start_iteration:end_iteration].copy()
            if df_subset.empty: continue
            
            first_config = df_subset.iloc[0]["GridConfigStarting"]
            species_array = parse_grid_config_string_to_species_array(first_config, 20)
            if species_array is None: continue
            
            N = species_array.shape[0]
            distinct_species = sorted(set(species_array.flatten()))
            colors = get_distinct_colors(len(distinct_species))
            color_map = {species: colors[i] for i, species in enumerate(distinct_species)}
            
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.set_xlim(-0.5, N-0.5)
            ax.set_ylim(-0.5, N-0.5)
            ax.set_aspect('equal')
            ax.set_title(f'Grid Animation: {table_label}')
            
            def animate(frame):
                ax.clear()
                ax.set_xlim(-0.5, N-0.5)
                ax.set_ylim(-0.5, N-0.5)
                ax.set_aspect('equal')
                ax.set_title(f'Grid Animation: {table_label} - Iteration {start_iteration + frame}')
                
                if frame < len(df_subset):
                    config = df_subset.iloc[frame]["GridConfigStarting"]
                    current_species = parse_grid_config_string_to_species_array(config, N)
                    
                    if current_species is not None:
                        for i in range(N):
                            for j in range(N):
                                species = current_species[i, j]
                                color = color_map.get(species, '#FFFFFF')
                                rect = Rectangle((j-0.4, N-1-i-0.4), 0.8, 0.8, 
                                               facecolor=color, edgecolor='black', linewidth=0.5)
                                ax.add_patch(rect)
                
                return []
            
            anim = animation.FuncAnimation(fig, animate, frames=len(df_subset), 
                                         interval=interval, blit=False, repeat=True)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error creating animation for {csv_file}: {str(e)}")
            continue

In [ ]:
# CELL 3: BATCH PROCESSING FUNCTIONS
def simple_batch_processor(*filenames, run_energy=True, run_clusters=True, run_grids=True, run_moves=True, run_animations=False):
    """Simple batch processor for copy-paste filenames"""
    if not filenames: return
    
    csv_files = [f"{base_name}.csv" for base_name in filenames]
    table_labels = list(filenames)
    curve_colours = get_distinct_colors(len(csv_files))
    
    existing_files = [f for f in csv_files if Path(f).is_file()]
    existing_labels = [table_labels[i] for i, f in enumerate(csv_files) if Path(f).is_file()]
    existing_colours = [curve_colours[i] for i, f in enumerate(csv_files) if Path(f).is_file()]
    
    if not existing_files: return
    
    if run_energy:
        energy_analysis_driver(existing_files, existing_labels, existing_colours)
    
    if run_clusters:
        cluster_analysis_driver(existing_files, existing_labels, existing_colours)
    
    if run_grids:
        grid_visualization_driver(existing_files, existing_labels)
    
    if run_moves:
        moves_analysis_driver(existing_files, existing_labels, existing_colours)
    
    if run_animations:
        animation_driver(existing_files, existing_labels)

def combinatoric_processor(BJ: str, include_kawasaki: bool = True, base_variants: List[str] = None,
                         signifiers: List[List[str]] = None, cluster_types: List[str] = None,
                         run_energy=True, run_clusters=True, run_grids=True, run_moves=True, run_animations=False):
    """Combinatoric processor for parameter sweeps"""
    
    base_variants = base_variants or ["DR"]
    signifiers = signifiers or [["tol0.50"], ["scp0.50"]]
    cluster_types = cluster_types or ["1", "2"]
    
    csv_files = []
    
    if include_kawasaki:
        csv_files.append(f"BJ={BJ}_Kawasaki_0_tol0.00_scp0.00.csv")
    
    for stem in base_variants:
        for ct in cluster_types:
            for sig1 in signifiers[0]:
                for sig2 in signifiers[1]:
                    csv_files.append(f"BJ={BJ}_{stem}_{ct}_{sig1}_{sig2}.csv")
    
    table_labels = [f.replace('.csv', '') for f in csv_files]
    curve_colours = get_distinct_colors(len(csv_files))
    
    existing_files = [f for f in csv_files if Path(f).is_file()]
    existing_labels = [table_labels[i] for i, f in enumerate(csv_files) if Path(f).is_file()]
    existing_colours = [curve_colours[i] for i, f in enumerate(csv_files) if Path(f).is_file()]
    
    if not existing_files: return
    
    if run_energy:
        energy_analysis_driver(existing_files, existing_labels, existing_colours, f"Combinatoric Analysis BJ={BJ}")
    
    if run_clusters:
        cluster_analysis_driver(existing_files, existing_labels, existing_colours)
    
    if run_grids:
        grid_visualization_driver(existing_files, existing_labels)
    
    if run_moves:
        moves_analysis_driver(existing_files, existing_labels, existing_colours)
    
    if run_animations:
        animation_driver(existing_files, existing_labels)

In [ ]:
# CELL 4: EXECUTION WITH BOOLEAN SWITCHES
USE_SIMPLE_BATCH = True
USE_COMBINATORIC = False

if USE_SIMPLE_BATCH:
    # Copy and paste your filenames here (without .csv extension)
    filenames_to_analyze = [
        "BJ=-0.44_Kawasaki_0_tol0.00_scp0.50",
        "BJ=-0.44_DR_1_tol0.50_scp0.50",
        "BJ=-0.44_DR_2_tol0.50_scp0.50"
    ]
    
    simple_batch_processor(
        *filenames_to_analyze,
        run_energy=True,      # Energy analysis and convergence
        run_clusters=True,    # Cluster size evolution
        run_grids=True,       # Static grid visualization
        run_moves=True,       # Move statistics
        run_animations=False  # Grid animations (requires ffmpeg)
    )

if USE_COMBINATORIC:
    combinatoric_processor(
        BJ="-0.44",
        include_kawasaki=True,
        base_variants=["DR", "NewMC"],
        signifiers=[["tol0.50", "tol0.25"], ["scp0.50", "scp0.25"]],
        cluster_types=["1", "2"],
        run_energy=True,
        run_clusters=True,
        run_grids=False,
        run_moves=True,
        run_animations=False
    )